# MMF Post-Processing & Evaluation — Reproducibility Notebook
Auto-generated by mmf-agent after interactive `/post-process-and-evaluate` session.

This notebook records the evaluation queries and best-model selection logic
and can be re-run to reproduce the same evaluation summary tables.

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
metric = "{metric}"

In [ ]:
eval_table = f"{catalog}.{schema}.{use_case}_evaluation_output"
score_table = f"{catalog}.{schema}.{use_case}_scoring_output"

eval_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {eval_table}").collect()[0]["cnt"]
score_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {score_table}").collect()[0]["cnt"]
print(f"Evaluation output: {eval_count:,} rows in {eval_table}")
print(f"Scoring output:    {score_count:,} rows in {score_table}")
assert eval_count > 0, f"Evaluation table {eval_table} is empty"
assert score_count > 0, f"Scoring table {score_table} is empty"

In [ ]:
eval_table = f"{catalog}.{schema}.{use_case}_evaluation_output"
score_table = f"{catalog}.{schema}.{use_case}_scoring_output"
best_forecastable_table = f"{catalog}.{schema}.{use_case}_best_models_forecastable"
best_table = f"{catalog}.{schema}.{use_case}_best_models"

spark.sql(f"""
CREATE OR REPLACE TABLE {best_forecastable_table} AS
WITH latest_run AS (
  SELECT run_id
  FROM {eval_table}
  ORDER BY run_date DESC
  LIMIT 1
)
SELECT eval.unique_id, eval.model, eval.avg_metric, score.ds, score.y,
       'main_pipeline' AS forecast_source, eval.run_id
FROM (
  SELECT unique_id, model, run_id, avg_metric,
         RANK() OVER (PARTITION BY unique_id ORDER BY avg_metric ASC) AS rank
  FROM (
    SELECT e.unique_id, e.model, e.run_id, AVG(e.metric_value) AS avg_metric
    FROM {eval_table} AS e
    INNER JOIN latest_run AS lr ON e.run_id = lr.run_id
    GROUP BY e.unique_id, e.model, e.run_id
    HAVING AVG(e.metric_value) IS NOT NULL
  )
) AS eval
INNER JOIN {score_table} AS score
  ON eval.unique_id = score.unique_id AND eval.model = score.model AND eval.run_id = score.run_id
WHERE eval.rank = 1
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {best_table} AS
SELECT unique_id, model, avg_metric, ds, y, forecast_source, run_id
FROM {best_forecastable_table}
""")

best_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {best_table}").collect()[0]["cnt"]
print(f"Best model selection: {best_count:,} rows in {best_table}")

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"
summary_table = f"{catalog}.{schema}.{use_case}_evaluation_summary"

spark.sql(f"""
CREATE OR REPLACE TABLE {summary_table} AS
SELECT
    model,
    COUNT(*) AS wins_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS wins_pct,
    ROUND(AVG(avg_metric), 4) AS avg_smape,
    CAST(NULL AS DOUBLE) AS avg_wape
FROM {best_table}
GROUP BY model
ORDER BY wins_count DESC
""")

print("Model ranking:")
spark.sql(f"SELECT model, wins_count, wins_pct, avg_smape FROM {summary_table}").show(truncate=False)

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"

overall = spark.sql(f"""
SELECT
  COUNT(DISTINCT unique_id) AS total_series,
  COUNT(DISTINCT model) AS distinct_winning_models,
  ROUND(AVG(avg_metric), 4) AS overall_avg_metric
FROM {best_table}
""").collect()[0]

print(f"Overall forecast quality:")
print(f"  Total series evaluated:    {overall['total_series']}")
print(f"  Distinct winning models:   {overall['distinct_winning_models']}")
print(f"  Overall avg {metric}:        {overall['overall_avg_metric']}")

print(f"\nWorst-performing series:")
spark.sql(f"""
SELECT unique_id, model, avg_metric
FROM {best_table}
ORDER BY avg_metric DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"
profile_table = f"{catalog}.{schema}.{use_case}_series_profile"

try:
    spark.sql(f"SELECT 1 FROM {profile_table} LIMIT 1")
    print("Cross-reference with series profiling:")
    spark.sql(f"""
    SELECT
        b.model,
        p.forecastability_class,
        COUNT(*) AS series_count,
        ROUND(AVG(b.avg_metric), 4) AS avg_metric
    FROM {best_table} b
    LEFT JOIN {profile_table} p ON b.unique_id = p.unique_id
    GROUP BY b.model, p.forecastability_class
    ORDER BY p.forecastability_class, avg_metric
    """).show(truncate=False)
except Exception:
    print("Series profile table not found — skipping cross-reference.")